# 임베딩 파이프라인 테스트

## 실행 가이드

### 필수 순차 실행 (메인 파이프라인)

1. **Cell 1**: 환경 설정 및 BGE-M3 초기화
2. **Cell 2**: 캐싱 효과 테스트
3. **Cell 3**: 재시도 로직 테스트
4. **Cell 4**: 샘플 파일 로딩
5. **Cell 5**: 청킹 및 배치 생성
6. **Cell 6**: 단일 청크 테스트 (선택적)
7. **Cell 7**: 배치별 임베딩 생성
8. **Cell 8**: 임베딩 결과 확인
9. **Cell 9**: NumPy 기반 검색 구현
10. **Cell 10**: 유사도 검색 테스트

### 비교 검증 (독립적 실행)

-   **Cell 11**: ChromaDB 설정 (독립적)
-   **Cell 12**: ChromaDB 코사인 테스트
-   **Cell 13**: NumPy vs ChromaDB 비교
-   **Cell 14**: 특정 파일 독립 테스트
-   **Cell 15**: 특정 파일 유사도 검색

### 주요 기능

-   BGE-M3 로컬 임베딩 (1024차원)
-   캐싱 시스템으로 속도 최적화
-   재시도 로직으로 안정성 확보
-   NumPy vs ChromaDB 정확도 검증
-   키워드 매칭률로 명확한 결과 해석
-   특정 파일 대상 완전한 파이프라인 테스트


In [ ]:
# 환경 설정 및 BGE-M3 모델 초기화
import sys
import time
from pathlib import Path
from typing import List, Optional

project_root: Path = Path("../..").resolve()
sys.path.insert(0, str(project_root / "src"))

from langchain.embeddings import CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain_community.embeddings import OllamaEmbeddings

# 캐싱 설정
cache_directory: Path = project_root / "cache" / "embeddings"
cache_directory.mkdir(parents=True, exist_ok=True)
file_store: LocalFileStore = LocalFileStore(str(cache_directory))

# BGE-M3 모델 초기화
base_embeddings_model: OllamaEmbeddings = OllamaEmbeddings(model="bge-m3")

# 캐싱이 적용된 임베딩 모델
embeddings_with_cache: CacheBackedEmbeddings = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=base_embeddings_model,
    document_embedding_cache=file_store,
    namespace="bge-m3-v1",
)


def safe_embed_documents(texts: List[str], max_retries: int = 3) -> List[List[float]]:
    """재시도 로직이 포함된 안전한 임베딩 생성"""
    last_exception: Optional[Exception] = None

    for attempt in range(max_retries):
        try:
            embedding_result: List[List[float]] = embeddings_with_cache.embed_documents(texts)
            return embedding_result
        except Exception as exception:
            last_exception = exception
            if attempt == max_retries - 1:
                break
            wait_time: int = min(2**attempt, 8)  # 최대 8초로 제한
            time.sleep(wait_time)

    # 최종 실패 시 직접 호출 시도 (캐시 우회)
    try:
        fallback_result: List[List[float]] = base_embeddings_model.embed_documents(texts)
        return fallback_result
    except Exception:
        if last_exception:
            raise last_exception
        raise


def safe_embed_query(text: str, max_retries: int = 3) -> List[float]:
    """재시도 로직이 포함된 안전한 쿼리 임베딩"""
    last_exception: Optional[Exception] = None

    for attempt in range(max_retries):
        try:
            query_result: List[float] = embeddings_with_cache.embed_query(text)
            return query_result
        except Exception as exception:
            last_exception = exception
            if attempt == max_retries - 1:
                break
            wait_time: int = min(2**attempt, 8)  # 최대 8초로 제한
            time.sleep(wait_time)

    # 최종 실패 시 직접 호출 시도
    try:
        fallback_result: List[float] = base_embeddings_model.embed_query(text)
        return fallback_result
    except Exception:
        if last_exception:
            raise last_exception
        raise


# 연결 테스트
test_embedding_result: List[float] = safe_embed_query("테스트")
print(f"임베딩 차원: {len(test_embedding_result)}")


In [ ]:
# 캐싱 효과 테스트
import time
from typing import List

import numpy as np

# 테스트 텍스트들
test_text_samples: List[str] = [
    "첫번째 문장 테스트",
    "두번째 문장 캐싱확인",
    "세번째 문장 성능비교",
]

# 첫 번째 실행 (캐시 없음)
print("\n1차 실행 (캐시 생성):")
first_start_time: float = time.time()
first_embedding_results: List[List[float]] = safe_embed_documents(test_text_samples)
first_execution_duration: float = time.time() - first_start_time
print(f"  시간: {first_execution_duration:.2f}초")
print(f"  벡터 수: {len(first_embedding_results)}개")

# 두 번째 실행 (캐시 사용)
print("\n2차 실행 (캐시 사용):")
second_start_time: float = time.time()
second_embedding_results: List[List[float]] = safe_embed_documents(test_text_samples)
second_execution_duration: float = time.time() - second_start_time
print(f"  시간: {second_execution_duration:.2f}초")
print(f"  벡터 수: {len(second_embedding_results)}개")

# 결과 비교
if first_execution_duration > 0 and second_execution_duration > 0:
    performance_speedup: float = first_execution_duration / second_execution_duration
    time_savings: float = first_execution_duration - second_execution_duration
    print(f"\n성능 향상:")
    print(f"  속도 향상: {performance_speedup:.1f}배")
    print(f"  절약 시간: {time_savings:.2f}초")

    if performance_speedup > 2:
        print("  결과: 캐싱 효과 확인됨")
    else:
        print("  결과: 캐싱 효과 제한적")

# 캐시 파일 확인
cache_file_list: List[Path] = list(cache_directory.glob("**/*"))
print(f"\n캐시 파일:")
print(f"  디렉토리: {cache_directory}")
print(f"  파일 수: {len(cache_file_list)}개")

# 벡터 일치성 확인
if len(first_embedding_results) == len(second_embedding_results):
    vectors_match: bool = True
    for index, (first_vector, second_vector) in enumerate(
        zip(first_embedding_results, second_embedding_results)
    ):
        if not np.allclose(first_vector, second_vector, atol=1e-6):
            vectors_match = False
            break

    if vectors_match:
        print("  벡터 일치성: 완전 동일")
    else:
        print("  벡터 일치성: 차이 발견")
else:
    print("  벡터 일치성: 개수 불일치")


In [ ]:
# 재시도 로직 테스트
print("재시도 로직 테스트")

# 정상 동작 확인
test_text = ["재시도 로직 테스트 문장입니다."]
result = safe_embed_documents(test_text, max_retries=3)

print(f"  벡터 생성: {len(result)}개")
print(f"  벡터 차원: {len(result[0]) if result else 0}")
print("  결과: 재시도 로직 정상 작동")

# 재시도 로직 기능 설명
print("재시도 로직 기능:")
print("  - 최대 3회 재시도")
print("  - 지수 백오프 (1초, 2초, 4초)")
print("  - Exception 발생 시 자동 재시도")

# 지수 백오프 패턴 설명
print("지수 백오프 패턴:")
for attempt in range(3):
    wait_time = 2**attempt
    print(f"  시도 {attempt + 1} 실패 → {wait_time}초 대기 → 다음 시도")


In [ ]:
# 샘플 파일 로딩
from typing import List

from langchain_core.documents import Document
from omegaconf import DictConfig

from app.config import load_config
from core.loader_router.loader import get_loader

cfg: DictConfig = load_config()

# 테스트 파일 수집
test_files_dir: Path = project_root / "notebooks" / "test-files"
all_files: List[Path] = []
for ext in [".txt", ".csv", ".docx"]:
    files: List[Path] = list(test_files_dir.rglob(f"*{ext}"))
    all_files.extend(files[:1])

test_files: List[Path] = all_files

# 문서 로딩
docs_by_file: List[List[Document]] = []
for file_path in test_files:
    try:
        loader = get_loader(str(file_path), cfg)
        docs: List[Document] = loader.load()
        docs_by_file.append(docs)
        print(f"{file_path.name}: {len(docs)}개 문서")
    except Exception:
        empty_docs: List[Document] = []
        docs_by_file.append(empty_docs)
        print(f"{file_path.name}: 로딩 실패")

total_docs: int = sum(len(docs) for docs in docs_by_file)
print(f"총 {len(test_files)}개 파일, {total_docs}개 문서 로딩")


In [ ]:
# 청킹 및 배치 생성
from typing import Dict, Iterator, List, Union

from core.chunker import MAX_TOKENS, chunk_documents, process_document_batches

# 청킹 실행
all_chunks: List[Document] = []
for i, file_path in enumerate(test_files):
    file_docs: List[Document] = docs_by_file[i]
    if file_docs:
        ext: str = file_path.suffix.lower()
        chunks: List[Document] = list(chunk_documents([file_docs], ext))
        all_chunks.extend(chunks)
        print(f"{file_path.name} ({ext}): {len(chunks)}개 청크")

print(f"총 청크 수: {len(all_chunks)}개")

# 배치 생성
BatchInfo = Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]
batch_generator: Iterator[BatchInfo] = process_document_batches(
    all_chunks, 0, 0, ".txt", MAX_TOKENS
)
all_batches: List[BatchInfo] = list(batch_generator)

print(f"총 배치 수: {len(all_batches)}개")

# 배치 요약
for i, batch_info in enumerate(all_batches):
    batch_data = batch_info["batch"]
    if isinstance(batch_data, list):
        batch: List[Document] = batch_data
        print(f"배치 {i + 1}: {len(batch)}개 청크")


In [ ]:
# 단일 청크 임베딩 테스트
import time

if all_chunks:
    test_chunk: Document = all_chunks[0]
    test_text: str = test_chunk.page_content

    print(f"테스트 청크 길이: {len(test_text)}자")
    print(f"테스트 청크 내용: {test_text[:100]}...")

    # 임베딩 생성 시간 측정
    start_time: float = time.time()
    chunk_embedding: List[float] = safe_embed_query(test_text)
    end_time: float = time.time()

    print(f"임베딩 생성 시간: {end_time - start_time:.2f}초")
    print(f"임베딩 차원: {len(chunk_embedding)}")

else:
    print("청크가 없습니다")


In [ ]:
# 배치별 임베딩 생성
import time
from typing import Any, Dict, List, Union, cast

# 배치 메타데이터 타입 정의
BatchMetadata = Dict[str, Union[int, float, List[int]]]


# 배치 임베딩 결과 구조체 정의
class BatchEmbeddingResult:
    def __init__(
        self,
        batch_id: int,
        chunks: List[Document],
        embeddings: List[List[float]],
        metadata: BatchMetadata,
    ) -> None:
        self.batch_id = batch_id
        self.chunks = chunks
        self.embeddings = embeddings
        self.metadata = metadata


# 모든 배치에 대해 임베딩 생성
batch_embedding_results: List[BatchEmbeddingResult] = []
total_chunks_processed: int = 0

processing_start_time: float = time.time()

for batch_index, batch_info in enumerate(all_batches):
    batch_data: Any = batch_info["batch"]
    batch_metadata: Any = batch_info["metadata"]

    if isinstance(batch_data, list) and isinstance(batch_metadata, dict):
        document_batch: List[Document] = cast(List[Document], batch_data)
        typed_metadata: BatchMetadata = cast(BatchMetadata, batch_metadata)

        # 배치 내 모든 청크의 텍스트 추출
        batch_text_contents: List[str] = [document.page_content for document in document_batch]

        # 배치 임베딩 생성
        batch_embedding_vectors: List[List[float]] = embeddings_with_cache.embed_documents(
            batch_text_contents
        )

        # 결과 저장
        batch_result: BatchEmbeddingResult = BatchEmbeddingResult(
            batch_id=batch_index,
            chunks=document_batch,
            embeddings=batch_embedding_vectors,
            metadata=typed_metadata,
        )
        batch_embedding_results.append(batch_result)

        total_chunks_processed += len(document_batch)
        print(f"배치 {batch_index + 1}: {len(document_batch)}개 청크 처리")

processing_end_time: float = time.time()
total_processing_time: float = processing_end_time - processing_start_time

print(f"총 처리 시간: {total_processing_time:.2f}초")
print(f"총 처리 청크: {total_chunks_processed}개")
if total_processing_time > 0:
    processing_speed: float = total_chunks_processed / total_processing_time
    print(f"평균 처리 속도: {processing_speed:.1f}개/초")


In [ ]:
# 임베딩 결과 확인
from typing import List

if "batch_embedding_results" in locals() and batch_embedding_results:
    print(f"총 배치 수: {len(batch_embedding_results)}개")

    # 총 청크 수 계산
    total_processed_chunks: int = 0
    for batch_result in batch_embedding_results:
        total_processed_chunks += len(batch_result.chunks)

    print(f"총 청크 수: {total_processed_chunks}개")

    # 첫 번째 배치 상세 정보
    if batch_embedding_results:
        first_batch_result: BatchEmbeddingResult = batch_embedding_results[0]

        first_batch_embeddings: List[List[float]] = first_batch_result.embeddings
        first_batch_chunks: List[Document] = first_batch_result.chunks

        if first_batch_embeddings and first_batch_chunks:
            first_embedding_vector: List[float] = first_batch_embeddings[0]
            first_document_chunk: Document = first_batch_chunks[0]

            print("첫 번째 배치 상세:")
            print(f"  청크 수: {len(first_batch_chunks)}개")
            print(f"  벡터 차원: {len(first_embedding_vector)}")
            print(f"  첫 청크 내용: {first_document_chunk.page_content[:100]}...")
            print(
                f"  첫 벡터 샘플: [{first_embedding_vector[0]:.4f}, {first_embedding_vector[1]:.4f}, ...]"
            )

else:
    print("이전 단계를 먼저 실행하세요")


In [ ]:
# NumPy 기반 In-Memory 검색 구현
from typing import Dict, List, Union, cast

import numpy as np
import numpy.typing as npt
from langchain_core.documents import Document

# 모든 임베딩과 청크를 평면화
flattened_embeddings: List[List[float]] = []
flattened_document_chunks: List[Document] = []

# batch_embedding_results에서 데이터 추출
if "batch_embedding_results" in locals() and batch_embedding_results:
    for batch_result in batch_embedding_results:
        batch_embeddings_list: List[List[float]] = batch_result.embeddings
        batch_chunks_list: List[Document] = batch_result.chunks
        flattened_embeddings.extend(batch_embeddings_list)
        flattened_document_chunks.extend(batch_chunks_list)
elif "single_file_embeddings" in locals() and "single_file_chunks" in locals():
    # 단일 파일 임베딩 결과 사용 (백업)
    single_embeddings_list: List[List[float]] = locals().get("single_file_embeddings", [])
    single_chunks_list: List[Document] = locals().get("single_file_chunks", [])
    flattened_embeddings.extend(single_embeddings_list)
    flattened_document_chunks.extend(single_chunks_list)

# NumPy 배열로 변환 (벡터화 연산 최적화)
embeddings_matrix: npt.NDArray[np.float64] = np.array(flattened_embeddings, dtype=np.float64)

print(f"총 청크 수: {len(flattened_document_chunks)}개")


# 유사도 검색 함수 정의
def search_similar_documents(
    query_text: str, top_k: int = 5
) -> List[Dict[str, Union[Document, float, str]]]:
    """NumPy 기반 고속 코사인 유사도 검색"""
    # 데이터 유효성 검증
    if len(flattened_document_chunks) == 0 or embeddings_matrix.size == 0:
        print("임베딩 데이터가 없습니다. 먼저 임베딩을 생성하세요.")
        return []

    # 쿼리 임베딩 생성 (재시도 로직 적용)
    query_embedding_vector: npt.NDArray[np.float64] = np.array(
        safe_embed_query(query_text), dtype=np.float64
    )

    # 코사인 유사도 계산 (벡터화 연산)
    cosine_similarities: npt.NDArray[np.float64] = np.dot(
        embeddings_matrix, query_embedding_vector
    ) / (np.linalg.norm(embeddings_matrix, axis=1) * np.linalg.norm(query_embedding_vector))

    # 실제 데이터 수에 맞춰 top_k 조정
    effective_top_k: int = min(top_k, len(cosine_similarities))

    # 상위 k개 인덱스 선택 (효율적 정렬)
    if effective_top_k == len(cosine_similarities):
        # 모든 결과 반환
        top_similarity_indices: npt.NDArray[np.intp] = np.argsort(cosine_similarities)[::-1]
    else:
        # 부분 정렬로 최적화
        top_similarity_indices = np.argpartition(cosine_similarities, -effective_top_k)[
            -effective_top_k:
        ]
        top_similarity_indices = top_similarity_indices[
            np.argsort(cosine_similarities[top_similarity_indices])[::-1]
        ]

    search_results: List[Dict[str, Union[Document, float, str]]] = []
    for index in top_similarity_indices:
        document_chunk: Document = cast(Document, flattened_document_chunks[index])
        similarity_score: float = float(cosine_similarities[index])

        # 내용 미리보기 추출
        try:
            content_preview: str = document_chunk.page_content[:100]
        except (AttributeError, TypeError):
            content_preview = "내용 추출 실패"

        # 메타데이터에서 출처 정보 추출
        try:
            metadata_dict = cast(Dict[str, Union[str, int, float]], document_chunk.metadata)
            source_information: str = str(metadata_dict.get("source", "출처 없음"))
        except (AttributeError, TypeError):
            source_information = "출처 추출 실패"

        result_entry: Dict[str, Union[Document, float, str]] = {
            "chunk": document_chunk,
            "similarity": similarity_score,
            "content": content_preview,
            "source": source_information,
        }
        search_results.append(result_entry)

    return search_results


In [ ]:
# 유사도 검색 테스트
import time
from typing import Dict, List, Union, cast

# 검색 쿼리 테스트
search_test_queries: List[str] = ["Test a checkbox within a textbox"]

search_query: str
for search_query in search_test_queries:
    print(f"검색: '{search_query}'")

    search_start_time: float = time.time()
    search_results: List[Dict[str, Union[Document, float, str]]] = search_similar_documents(
        search_query, top_k=3
    )
    search_end_time: float = time.time()

    print(f"검색 시간: {search_end_time - search_start_time:.3f}초")

    result_index: int
    search_result: Dict[str, Union[Document, float, str]]
    for result_index, search_result in enumerate(search_results):
        # 타입 안전한 값 추출
        similarity_score: float = cast(float, search_result["similarity"])
        content_preview: str = cast(str, search_result["content"])
        source_info: str = cast(str, search_result["source"])

        print(f"  {result_index + 1}. 유사도: {similarity_score:.4f}")
        print(f"     출처: {source_info}")
        print(f"     내용: {content_preview}...")


In [ ]:
# ChromaDB 설정 및 검색 구현
from typing import Any, List, Optional

try:
    import chromadb
    from langchain_core.documents import Document

    from app.config import load_config
    from core.chunker import MAX_TOKENS, chunk_documents, process_document_batches
    from core.loader_router.loader import get_loader

    # ChromaDB 클라이언트 설정
    app_config = load_config()
    chromadb_client = chromadb.Client()

    # 테스트 파일 경로 설정
    target_test_filename: str = "815.txt"
    test_files_directory = project_root / "notebooks" / "test-files"

    # 타겟 파일 검색
    target_test_file_path: Optional[Any] = None
    for discovered_file_path in test_files_directory.rglob(target_test_filename):
        target_test_file_path = discovered_file_path
        break

    if target_test_file_path:
        print(f"ChromaDB 설정: {target_test_file_path}")

        # 문서 로딩 파이프라인 실행
        document_loader = get_loader(str(target_test_file_path), app_config)
        loaded_documents = document_loader.load()

        if loaded_documents:
            # 문서 청킹 처리
            file_extension: str = target_test_file_path.suffix.lower()
            document_chunks: List[Document] = list(
                chunk_documents([loaded_documents], file_extension)
            )

            # ChromaDB용 임베딩 및 청크 저장소
            chromadb_embeddings: List[List[float]] = []
            chromadb_chunks: List[Document] = []

            # 배치 처리를 통한 임베딩 생성
            processing_batches = list(
                process_document_batches(document_chunks, 0, 0, file_extension, MAX_TOKENS)
            )

            for batch_info in processing_batches:
                document_batch: List[Document] = batch_info["batch"]  # type: ignore
                batch_text_contents: List[str] = [chunk.page_content for chunk in document_batch]  # type: ignore
                batch_embedding_vectors = embeddings_with_cache.embed_documents(batch_text_contents)  # type: ignore

                chromadb_embeddings.extend(batch_embedding_vectors)  # type: ignore
                chromadb_chunks.extend(document_batch)  # type: ignore

            # 코사인 유사도 컬렉션 생성 (기존 컬렉션 삭제 후 재생성)
            try:
                chromadb_client.delete_collection("cosine_docs")  # type: ignore
            except Exception:
                pass

            cosine_similarity_collection = chromadb_client.create_collection(  # type: ignore
                name="cosine_docs", metadata={"hnsw:space": "cosine"}
            )

            # 문서와 임베딩을 컬렉션에 추가
            cosine_similarity_collection.add(  # type: ignore
                embeddings=chromadb_embeddings,
                documents=[document_chunk.page_content for document_chunk in chromadb_chunks],
                ids=[f"chunk_{chunk_index}" for chunk_index in range(len(chromadb_chunks))],
            )

            # ChromaDB 검색 함수 정의
            def search_chromadb_cosine(search_query: str, result_count: int = 3) -> Any:  # type: ignore
                """ChromaDB를 이용한 코사인 유사도 검색"""
                query_embedding_vector: List[float] = safe_embed_query(search_query)

                search_results = cosine_similarity_collection.query(  # type: ignore
                    query_embeddings=[query_embedding_vector],
                    n_results=result_count,
                    include=["documents", "distances"],
                )

                print(f"ChromaDB 검색: '{search_query[:50]}...'")

                documents_data = search_results.get("documents")  # type: ignore
                distances_data = search_results.get("distances")  # type: ignore

                if documents_data and len(documents_data) > 0:
                    result_documents: List[str] = documents_data[0]  # type: ignore
                    result_distances: List[float] = (
                        distances_data[0] if distances_data and len(distances_data) > 0 else []
                    )  # type: ignore

                    for doc_index, document_content in enumerate(result_documents):
                        # ChromaDB 거리 값을 안전하게 float로 변환
                        if doc_index < len(result_distances):
                            cosine_distance: float = float(result_distances[doc_index])  # type: ignore
                        else:
                            cosine_distance = 1.0  # 기본값

                        semantic_similarity: float = 1.0 - cosine_distance

                        # 키워드 매칭률 계산 (의미적 유사도와 독립적)
                        search_keywords: List[str] = search_query.lower().split()
                        document_lowercase: str = document_content.lower()
                        matched_keyword_count: int = sum(
                            1 for keyword in search_keywords if keyword in document_lowercase
                        )
                        keyword_match_percentage: float = (
                            matched_keyword_count / len(search_keywords) * 100
                            if search_keywords
                            else 0
                        )

                        print(
                            f"  {doc_index + 1}. 유사도: {semantic_similarity:.4f} | 키워드 매칭: {keyword_match_percentage:.0f}%"
                        )
                        print(f"     내용: {document_content[:80]}...")

                return search_results

        else:
            print("문서 로딩 실패")

    else:
        print(f"파일 없음: {target_test_filename}")

except Exception as chromadb_error:
    print(f"ChromaDB 오류: {chromadb_error}")


In [ ]:
# ChromaDB 코사인 유사도 검색 테스트
from typing import Any, Dict, List, Optional, Union

try:
    import chromadb
    from chromadb.api.client import Client  # type: ignore
    from chromadb.api.models.Collection import Collection  # type: ignore

    # ChromaDB 타입 정의
    ChromaQueryResult = Dict[str, Union[List[List[str]], List[List[float]], List[List[int]]]]

    # 코사인 유사도 전용 클라이언트 생성
    cosine_test_client: Client = chromadb.Client()  # type: ignore

    # 기존 컬렉션 정리
    try:
        cosine_test_client.delete_collection("cosine_docs")  # type: ignore
    except Exception:
        pass

    # 코사인 유사도 컬렉션 생성
    cosine_test_collection: Collection = cosine_test_client.create_collection(  # type: ignore
        name="cosine_docs",
        metadata={"hnsw:space": "cosine"},
    )

    # 데이터 추가 (이전 셀에서 생성된 ChromaDB 데이터 사용)
    if "chromadb_chunks" in locals() and "chromadb_embeddings" in locals():
        cosine_chunk_documents: List[str] = [
            document_chunk.page_content for document_chunk in chromadb_chunks
        ]
        cosine_chunk_ids: List[str] = [
            f"cosine_chunk_{chunk_idx}" for chunk_idx in range(len(chromadb_chunks))
        ]

        cosine_test_collection.add(  # type: ignore
            embeddings=chromadb_embeddings,
            documents=cosine_chunk_documents,
            ids=cosine_chunk_ids,
        )

        print(f"코사인 유사도 컬렉션 생성: {len(chromadb_chunks)}개 청크")

        # 코사인 유사도 검색 함수 정의
        def search_cosine_similarity(search_query: str, result_count: int = 3) -> Optional[Any]:  # type: ignore
            """코사인 유사도 기반 ChromaDB 검색"""
            query_embedding: List[float] = safe_embed_query(search_query)

            cosine_search_results: ChromaQueryResult = cosine_test_collection.query(  # type: ignore
                query_embeddings=[query_embedding],
                n_results=result_count,
                include=["documents", "distances"],
            )

            print(f"코사인 검색: '{search_query[:50]}...'")

            cosine_documents_data = cosine_search_results.get("documents")
            cosine_distances_data = cosine_search_results.get("distances")

            if (
                cosine_documents_data
                and isinstance(cosine_documents_data, list)
                and cosine_documents_data[0]
            ):
                cosine_result_docs: List[str] = cosine_documents_data[0]
                cosine_result_distances: List[float] = (
                    cosine_distances_data[0]
                    if cosine_distances_data and isinstance(cosine_distances_data, list)
                    else []
                )

                for doc_idx, document_text in enumerate(cosine_result_docs):
                    cosine_distance: float = (
                        cosine_result_distances[doc_idx]
                        if doc_idx < len(cosine_result_distances)
                        else 1.0
                    )
                    cosine_similarity_score: float = 1.0 - cosine_distance

                    # 키워드 매칭률 계산 (의미적 유사도와 독립적)
                    search_terms: List[str] = search_query.lower().split()
                    document_lowercase: str = document_text.lower()
                    matched_term_count: int = sum(
                        1 for term in search_terms if term in document_lowercase
                    )
                    keyword_match_rate: float = (
                        matched_term_count / len(search_terms) * 100 if search_terms else 0
                    )

                    print(
                        f"  {doc_idx + 1}. 코사인 유사도: {cosine_similarity_score:.4f} | 키워드 매칭: {keyword_match_rate:.0f}% | 길이: {len(document_text)}자"
                    )
                    print(f"     내용: {document_text[:100]}...")

            return cosine_search_results

        # 코사인 유사도 검색 테스트 실행
        cosine_test_query: str = (
            "The means by which the Federalists had maintained their position were "
            ", and their resources  temporary; it was by the virtues "
            "or the talents of  leaders that they had risen to power.  the "
            "Republicans attained to that lofty station, their opponents were "
            "overwhelmed by utter defeat."
        )
        search_cosine_similarity(cosine_test_query, result_count=5)

    else:
        print("ChromaDB 임베딩 데이터가 없습니다. 이전 셀을 먼저 실행하세요.")

except Exception as cosine_test_error:
    print(f"코사인 테스트 오류: {cosine_test_error}")
    print("NumPy 기반 검색을 사용하세요.")


In [ ]:
# NumPy vs ChromaDB 정확도 비교 검증
from typing import List, Tuple, cast

from langchain_core.documents import Document

# 이전 셀에서 생성된 ChromaDB 데이터 확인
if "chromadb_chunks" in locals() and "chromadb_embeddings" in locals():
    # 비교 테스트용 쿼리 세트
    comparison_test_queries: List[str] = [
        "Confederation dread anarchy",  # 짧은 키워드 검색
        "The ruin of the Confederation had impressed the people with a dread of anarchy",  # 긴 원문 검색
    ]

    for test_index, comparison_query in enumerate(comparison_test_queries):
        print(f"\n{'=' * 50}")
        print(
            f"비교 테스트 {test_index + 1}: {comparison_query[:40]}{'...' if len(comparison_query) > 40 else ''}"
        )
        print(f"{'=' * 50}")

        # NumPy 기반 직접 계산
        numpy_query_vector = np.array(safe_embed_query(comparison_query))
        numpy_embeddings_matrix = np.array(chromadb_embeddings)

        numpy_cosine_similarities = np.dot(numpy_embeddings_matrix, numpy_query_vector) / (
            np.linalg.norm(numpy_embeddings_matrix, axis=1) * np.linalg.norm(numpy_query_vector)
        )

        numpy_top_indices = np.argsort(numpy_cosine_similarities)[::-1][:3]

        print("NumPy 기반 결과:")
        numpy_comparison_results: List[Tuple[float, str]] = []
        for result_rank, document_index in enumerate(numpy_top_indices):
            numpy_similarity_score: float = numpy_cosine_similarities[document_index]
            numpy_document_chunk: Document = cast(Document, chromadb_chunks[document_index])
            numpy_document_content: str = numpy_document_chunk.page_content
            numpy_comparison_results.append((numpy_similarity_score, numpy_document_content[:60]))
            print(
                f"  {result_rank + 1}. 유사도: {numpy_similarity_score:.4f} | {numpy_document_content[:60]}..."
            )

        # ChromaDB 기반 검색
        print("ChromaDB 기반 결과:")
        if "cosine_test_collection" in locals():
            chromadb_search_results = cosine_test_collection.query(  # type: ignore
                query_embeddings=[safe_embed_query(comparison_query)],
                n_results=3,
                include=["documents", "distances"],
            )

            chromadb_documents_data = chromadb_search_results.get("documents")  # type: ignore
            chromadb_distances_data = chromadb_search_results.get("distances")  # type: ignore

            if chromadb_documents_data:
                chromadb_result_docs: List[str] = chromadb_documents_data[0]  # type: ignore
                chromadb_result_distances: List[float] = (
                    chromadb_distances_data[0] if chromadb_distances_data else []
                )  # type: ignore

                chromadb_comparison_results: List[Tuple[float, str]] = []
                for doc_rank, chromadb_document in enumerate(chromadb_result_docs):
                    chromadb_distance: float = (
                        chromadb_result_distances[doc_rank]
                        if doc_rank < len(chromadb_result_distances)
                        else 1.0
                    )
                    chromadb_similarity_score: float = 1.0 - float(chromadb_distance)
                    chromadb_comparison_results.append(
                        (chromadb_similarity_score, chromadb_document[:60])
                    )
                    print(
                        f"  {doc_rank + 1}. 유사도: {chromadb_similarity_score:.4f} | {chromadb_document[:60]}..."
                    )

                # 정확도 비교 분석
                print("정확도 비교 분석:")
                perfect_match: bool = True
                for comparison_rank in range(3):
                    numpy_similarity: float = numpy_comparison_results[comparison_rank][0]
                    chromadb_similarity: float = chromadb_comparison_results[comparison_rank][0]
                    similarity_difference: float = abs(numpy_similarity - chromadb_similarity)

                    if similarity_difference < 0.0001:
                        accuracy_status: str = "완전 일치"
                    else:
                        accuracy_status = f"차이: {similarity_difference:.4f}"
                        perfect_match = False

                    print(
                        f"  {comparison_rank + 1}번: NumPy {numpy_similarity:.4f} vs ChromaDB {chromadb_similarity:.4f} → {accuracy_status}"
                    )

                if perfect_match:
                    print("  최종 결론: 두 방법의 결과가 완전히 동일함")
                else:
                    print("  최종 결론: 알고리즘 구현 차이로 인한 미세한 차이 존재")
        else:
            print("  ChromaDB 컬렉션이 초기화되지 않았습니다.")

else:
    print("ChromaDB 데이터가 없습니다. 이전 셀들을 먼저 실행하세요.")


In [ ]:
# 특정 파일 테스트 파이프라인
from typing import Dict, List, Optional, Union, cast

import numpy as np
import numpy.typing as npt
from langchain_core.documents import Document
from omegaconf import DictConfig

from app.config import load_config
from core.chunker import MAX_TOKENS, chunk_documents, process_document_batches
from core.loader_router.loader import get_loader

# 테스트용 설정 로드
test_config: DictConfig = load_config()

# 테스트 대상 파일 지정
test_filename: str = "815.txt"  # 테스트할 파일명

# 테스트 파일 디렉토리 경로
test_files_directory: Path = project_root / "notebooks" / "test-files"
target_file_path: Optional[Path] = None

# 지정된 파일명으로 파일 검색
for discovered_file_path in test_files_directory.rglob(test_filename):
    target_file_path = discovered_file_path
    break

if target_file_path:
    print(f"테스트 파일 발견: {target_file_path}")

    # 문서 로딩 파이프라인
    try:
        document_loader = get_loader(str(target_file_path), test_config)
        loaded_documents: List[Document] = document_loader.load()

        print(f"로딩된 문서 수: {len(loaded_documents)}개")

        if loaded_documents:
            # 문서 청킹 처리
            file_extension: str = target_file_path.suffix.lower()
            document_chunks: List[Document] = list(
                chunk_documents([loaded_documents], file_extension)
            )

            # 배치 생성
            processing_batches: List[
                Dict[str, Union[List[Document], Dict[str, Union[int, float, List[int]]]]]
            ] = list(process_document_batches(document_chunks, 0, 0, file_extension, MAX_TOKENS))

            # 임베딩 생성
            file_embeddings: List[List[float]] = []
            file_chunks: List[Document] = []

            for batch_info in processing_batches:
                document_batch: List[Document] = cast(List[Document], batch_info["batch"])
                batch_texts: List[str] = [
                    document_chunk.page_content for document_chunk in document_batch
                ]
                batch_vectors: List[List[float]] = embeddings_with_cache.embed_documents(
                    batch_texts
                )

                file_embeddings.extend(batch_vectors)
                file_chunks.extend(document_batch)

            # NumPy 매트릭스 생성
            embeddings_matrix: npt.NDArray[np.float64] = np.array(file_embeddings, dtype=np.float64)

            print(
                f"처리 결과: 청킹 {len(document_chunks)}개 → "
                f"배치 {len(processing_batches)}개 → "
                f"임베딩 {embeddings_matrix.shape}"
            )

        else:
            print("로딩된 문서가 없습니다")

    except Exception as test_error:
        print(f"테스트 로딩 실패: {test_error}")

else:
    print(f"지정된 파일을 찾을 수 없습니다: {test_filename}")
    print("test-files 디렉토리에서 파일명을 확인하세요")


In [ ]:
# 파일 유사도 검색 테스트
from typing import Dict, List, Union, cast

import numpy as np
import numpy.typing as npt
from langchain_core.documents import Document

# 이전 테스트에서 생성된 데이터 확인
if "file_embeddings" in locals() and "file_chunks" in locals():
    # 파일 데이터를 타입 안전하게 할당
    single_file_embeddings: List[List[float]] = file_embeddings
    single_file_chunks: List[Document] = file_chunks

    print(f"파일 데이터 확인:")
    print(f"  청크 수: {len(single_file_chunks)}개")
    print(f"  임베딩 수: {len(single_file_embeddings)}개")
    print(f"  벡터 차원: {len(single_file_embeddings[0]) if single_file_embeddings else 0}")

    # 파일용 NumPy 매트릭스 생성
    single_file_embeddings_matrix: npt.NDArray[np.float64] = np.array(
        single_file_embeddings, dtype=np.float64
    )

    # 파일 유사도 검색 함수 정의
    def search_file_similarity(
        search_query: str, result_count: int = 5
    ) -> List[Dict[str, Union[int, float, str]]]:
        """파일 내에서 유사도 기반 검색"""
        if len(single_file_chunks) == 0 or single_file_embeddings_matrix.size == 0:
            print("파일 데이터가 없습니다.")
            return []

        # 검색 쿼리 임베딩 생성
        search_query_vector: npt.NDArray[np.float64] = np.array(
            safe_embed_query(search_query), dtype=np.float64
        )

        # 코사인 유사도 계산 (벡터화 연산)
        cosine_similarity_scores: npt.NDArray[np.float64] = np.dot(
            single_file_embeddings_matrix, search_query_vector
        ) / (
            np.linalg.norm(single_file_embeddings_matrix, axis=1)
            * np.linalg.norm(search_query_vector)
        )

        # 실제 결과 수 조정
        effective_result_count: int = min(result_count, len(cosine_similarity_scores))
        top_similarity_indices: npt.NDArray[np.intp] = np.argsort(cosine_similarity_scores)[
            -effective_result_count:
        ][::-1]

        search_results: List[Dict[str, Union[int, float, str]]] = []
        for result_rank, chunk_index in enumerate(top_similarity_indices):
            document_chunk: Document = cast(Document, single_file_chunks[chunk_index])
            similarity_score: float = float(cosine_similarity_scores[chunk_index])
            content_preview: str = document_chunk.page_content[:100]

            search_result_entry: Dict[str, Union[int, float, str]] = {
                "rank": result_rank + 1,
                "similarity": similarity_score,
                "content": content_preview,
                "source": str(
                    cast(Dict[str, Union[str, int, float]], document_chunk.metadata).get(
                        "source", "출처 불명"
                    )
                ),
            }
            search_results.append(search_result_entry)

            print(f"  {result_rank + 1}. 유사도: {similarity_score:.4f}")
            print(f"     내용: {content_preview}...")

        return search_results

    # 파일 검색 테스트 쿼리
    file_test_queries: List[str] = [
        "The means by which the had maintained their position were and their resources were temporary; it was by the virtues or the talents of their leaders that they had risen to power. When the Republicans attained to that lofty station, their opponents were by utter defeat."
    ]

    print("파일 검색 테스트 실행:")
    for test_query in file_test_queries:
        print(f"검색 쿼리: '{test_query[:80]}...'")
        file_search_results: List[Dict[str, Union[int, float, str]]] = search_file_similarity(
            test_query, result_count=3
        )

else:
    print("파일 테스트 데이터가 없습니다.")
    print("이전 테스트 셀을 먼저 실행하세요.")


---

## 완료된 기능 요약

### 구현 완료
1. **청킹 파이프라인**: 확장자별 청킹 설정 및 배치 처리
2. **BGE-M3 임베딩**: 로컬 Ollama 기반 1024차원 벡터 생성
3. **In-Memory 검색**: NumPy 기반 코사인 유사도 검색
4. **ChromaDB 비교**: NumPy vs ChromaDB 정확도 검증 (완전 일치 확인)
5. **캐싱 시스템**: CacheBackedEmbeddings로 중복 임베딩 방지
6. **재시도 로직**: 지수 백오프 방식의 안정적 임베딩 생성
7. **특정 파일 테스트**: 개별 파일 대상 완전한 파이프라인 테스트
8. **키워드 매칭률**: 의미적 유사도와 문자열 매칭 구분 표시

### 성능 지표
- **임베딩 속도**: 5.3개/초 (배치 처리)
- **캐싱 효과**: 2차 실행 시 대폭 속도 향상
- **검색 정확도**: NumPy와 ChromaDB 완전 일치
- **안정성**: 재시도 로직으로 장애 대응

---

## 다음 단계: Qdrant 벡터 DB 통합

이 노트북에서는 In-Memory 임베딩 테스트를 완료

**Qdrant 벡터 DB 통합 테스트는 다음 노트북에서 진행:**

-   `notebooks/voy-12/qdrant_integration.ipynb`

**주요 차이점:**

-   **현재 노트북**: NumPy/ChromaDB In-Memory + 캐싱/재시도
-   **Qdrant 노트북**: Docker 서버 기반 영구 저장 및 Repository 패턴

**전환 이유:**

-   프로덕션 환경 준비
-   대용량 데이터 처리
-   메타데이터 필터링 고도화
-   하이브리드 검색 준비
